In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests 
import os
import logging
import time
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# Globals

raw_data_dir = "../data/raw"
proc_data_dir = "../data/processed"

# EDIT THIS!!!
headers = {
    "User-Agent": "First Last email@gmail.com"
}

In [7]:
# Setting up logger
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, filename = "log.log", filemode = "w", 
                    format = "%(asctime)s - %(levelname)s - %(message)s")

## Creating Datasets

In [ ]:
# File downloader

def download_file(url: str, filename: str, headers: dict = {}) -> bool:
    save_path = os.path.join(raw_data_dir, filename)

    if os.path.exists(save_path):
        logger.info(f"{filename} already exists. Skipping download.")
        return True
    
    try:
        logger.info(f"Downloading {filename} from web...")

        response = requests.get(url, headers = headers, stream = True)

        response.raise_for_status()

        with open(save_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        logger.info(f"Successfully saved to {save_path}")
        return True
    
    except Exception as e:
        logger.error(f"Failed to download {filename}: {e}")
        return False


In [14]:
download_file("https://www.sec.gov/files/company_tickers.json","company_tickers.json", headers)

True

In [ ]:
df = pd.read_json("../data/raw/company_tickers.json", orient = "index")
df["cik_str"] = df["cik_str"].astype(str).str.zfill(10)

display(df.head())

,cik_str,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0001652044,GOOGL,Alphabet Inc.
2,0000320193,AAPL,Apple Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC


In [ ]:
def fetch_single_comp_metrics(ticker: str) -> dict:

    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        return {
            "ticker": ticker,
            
            # Quantitative block for ML
            "forwardPE": info.get("forwardPE"),
            "ev_to_ebitda": info.get("enterpriseToEbitda"),
            "ebitda_margin": info.get("ebitdaMargins"),
            "debt_to_equity": info.get("debtToEquity"),
            
            # Qualitative block for ML
            "sector": info.get("sector", "Unknown"),
            "industry": info.get("industry", "Unknown"),
            "business_summary": info.get("longBusinessSummary", ""),
            
            # Data for valuation
            "ebitda": info.get("ebitda"),
            "total_cash": info.get("totalCash"),
            "total_debt": info.get("totalDebt"),
            "shares_outstanding": info.get("sharesOutstanding")
        }
    except Exception as e:
            logger.warning(f"Failed to fetch data for {ticker}: {e}")
            return None



In [ ]:
# Testing function
fetch_single_comp_metrics("NVDA")

{'ticker': 'NVDA',
 'forwardPE': 17.73965,
 'ev_to_ebitda': 35.888,
 'ebitda_margin': 0.61698,
 'debt_to_equity': 7.255,
 'sector': 'Technology',
 'industry': 'Semiconductors',
 'business_summary': "NVIDIA Corporation operates as a data center scale AI infrastructure company. The company operates through two segments, Compute & Networking, and Graphics segments. The Compute & Networking segment provides data center accelerated computing and networking platforms and artificial intelligence solutions and software, and automotive platforms and autonomous and electric vehicle solutions, including software. The Graphics segment offers GeForce GPUs for gaming and PCs; Quadro/NVIDIA RTX GPUs for enterprise workstation graphics. The company's products are used in gaming, professional visualization, data center, and automotive markets. The company sells its products to original equipment manufacturers, original device manufacturers, system integrators and distributors, independent software vend

In [68]:
def build_csv_comps_table(raw_data_path: str, output_csv: str, chunk_size: int = 50, limit: int = None):
        logger.info("Starting large data pull...")
        df_raw = pd.read_csv(raw_data_path)
        all_tickers = df_raw['failed_tickers'].tolist()

        if limit:
            all_tickers = all_tickers[:limit]
            logger.info(f"Test Mode: Only processing the first {limit} companies")
        
        # Check if a partial file already exists to resume
        start_index = 0
        if os.path.exists(output_csv):
            existing_df = pd.read_csv(output_csv)
            start_index = len(existing_df)
            logger.info(f"Found existing file. Resuming from ticker {start_index}...")

        for i in range(0, len(all_tickers), chunk_size):
            chunk = all_tickers[i : i + chunk_size]

            max_attempts = 3
            attempt = 0
            success = False

            while attempt < max_attempts and not success:
                chunk_data = []

                for ticker in tqdm(chunk, desc=f"Chunk {i//chunk_size}", leave=False):
                    metrics = fetch_single_comp_metrics(ticker)
                    if metrics:
                        chunk_data.append(metrics)

                if len(chunk_data) == 0:
                    attempt += 1
                    logger.warning(f"Chunk {i//chunk_size} failed. Attempt {attempt}/{max_attempts}. Entering 60s cooldown...")
                    time.sleep(60)
                else:
                    success = True

            
            # Save the chunk to the CSV 
            if success:
                chunk_df = pd.DataFrame(chunk_data)

                # Logic to account for missing chunk data
                successful_tickers = chunk_df['ticker'].tolist()
                missed_tickers = [t for t in chunk if t not in successful_tickers]
                
                # Dead letter queue (DQL)
                if missed_tickers:
                    dlq_df = pd.DataFrame({"failed_tickers": missed_tickers})
                    dlq_csv = "../data/processed/missed_tickers.csv"
                    dlq_df.to_csv(dlq_csv, mode='a', header=not os.path.exists(dlq_csv), index=False)
                    logger.info(f"Chunk {i//chunk_size}: {len(missed_tickers)} tickers missing. Saved to DLQ.")

                # If file exists, append without headers. Otherwise, write new
                chunk_df.to_csv(output_csv, mode=""a', header=not os.path.exists(output_csv), index=False)
                time.sleep(10) # Safety pause

            else:
                logger.error(f"CRITICAL: Chunk {i//chunk_size} failed after {max_attempts} attempts. Skipping chunk to keep pipeline alive.")
                dlq_df = pd.DataFrame({"failed_tickers": chunk})
                dlq_csv = "../data/processed/missed_tickers.csv"
                dlq_df.to_csv(dlq_csv, mode='a', header=not os.path.exists(dlq_csv), index=False)

            

In [69]:
build_csv_comps_table("../data/processed/missed_tickers.csv", "../data/raw/company_metrics.csv", 25)

In [70]:
pd.read_csv("../data/raw/company_metrics.csv") # 10279



,ticker,forwardPE,ev_to_ebitda,ebitda_margin,debt_to_equity,sector,industry,business_summary,ebitda,total_cash,total_debt,shares_outstanding
0,NVDA,17.739650,35.888,0.61698,7.255,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...,1.332300e+11,6.255600e+10,1.141200e+10,2.430000e+10
1,GOOGL,25.076876,26.757,0.37279,16.133,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...,1.501750e+11,1.268430e+11,6.699600e+10,5.822000e+09
2,AAPL,28.615034,25.736,0.35100,102.630,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ...",1.529020e+11,6.690700e+10,9.050900e+10,1.468114e+10
3,MSFT,21.752918,17.616,0.57377,31.539,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...,1.752590e+11,8.946200e+10,1.232780e+11,7.425629e+09
4,AMZN,26.442839,18.686,0.20327,43.435,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of...",1.457310e+11,1.230290e+11,1.785470e+11,1.075425e+10
...,...,...,...,...,...,...,...,...,...,...,...,...
10363,HAVAR,NaN,NaN,0.00000,NaN,NaN,NaN,Harvard Ave Acquisition Corporation focuses on...,NaN,9.652400e+05,3.317300e+05,NaN
10364,DMIIR,NaN,NaN,0.00000,NaN,NaN,NaN,Drugs Made In America Acquisition II Corp. doe...,NaN,3.150870e+05,0.000000e+00,NaN
10365,DMIIU,NaN,NaN,0.00000,NaN,Financial Services,Shell Companies,Drugs Made In America Acquisition II Corp. doe...,NaN,3.150870e+05,0.000000e+00,NaN
10366,APACU,NaN,NaN,0.00000,NaN,Financial Services,Shell Companies,StoneBridge Acquisition II Corporation does no...,NaN,5.038300e+05,2.200000e+01,NaN


In [ ]:
df = pd.read_csv("../data/processed/missed_tickers.csv")
all_tickers = df["failed_tickers"].tolist()
print(all_tickers)

['CYD', 'CNXC', 'HIMX', 'STBA', 'WD', 'BFC', 'NUVB', 'BIOT', 'GPAC', 'BRW', 'PNNT', 'SVAQ', 'FCCO', 'HCAC', 'VGZ', 'IRHO', 'OVLY', 'SEG', 'KRYXF', 'PAYS', 'GLATF', 'UCFI', 'EM', 'EDIT', 'WBX', 'TCBS', 'UCL', 'CDLX', 'CPPMF', 'EDSA', 'PETS', 'XLO', 'GLXZ', 'GRNQ', 'ALGS', 'RYM', 'OMEX', 'BREZ', 'SEOVF', 'KSEZ', 'QPRC', 'LRHC', 'AHRO', 'OMQS', 'JZXN', 'CBDY', 'LBUY', 'EVFM', 'SKFG', 'FECOF', 'ESMC', 'TLSS', 'NIVF', 'PFSA', 'SBEV', 'ISCO', 'MDCE', 'VHAI', 'AERA', 'ACRL', 'TGCB', 'VISL', 'CANN', 'IOBT', 'UCLE', 'EBZT', 'NMGX', 'MTTCF', 'GRST', 'NUVI', 'WDLF', 'FZMD', 'BWMG', 'RDAR', 'QH', 'ECIA', 'VCNX', 'UCASU', 'RSVRW', 'BFRGW', 'GLCP', 'CPPTL', 'GEGGL', 'HAVAU', 'HAVAR', 'DMIIR', 'DMIIU', 'APACU', 'APACR']


In [2]:
df = pd.read_csv("../data/raw/company_metrics.csv")

df = df.dropna(subset=["ebitda", "sector", "industry", "shares_outstanding", "business_summary"])
df["total_debt"] = df["total_debt"].fillna(0)
df["total_cash"] = df["total_cash"].fillna(0)

# Adding more robust debt_to_ebitda metric
df["debt_to_ebitda"] = df["total_debt"] / df["ebitda"]
df = df.drop(columns=["debt_to_equity"])

# Conditional fill to account for ev_to_ebita ratios that are because of negative EBDITA vs just missing
df.loc[df["ebitda"] <= 0, "ev_to_ebitda"] = df.loc[df["ebitda"] <= 0, "ev_to_ebitda"].fillna(0)
df["ev_to_ebitda"] = df.groupby("sector")["ev_to_ebitda"].transform(lambda x: x.fillna(x.median()))
df["forwardPE"] = df.groupby("sector")["forwardPE"].transform(lambda x: x.fillna(x.median()))
df = df.dropna(subset=["ev_to_ebitda", "forwardPE"])

df = df.reset_index(drop=True)

df.to_parquet("../data/processed/company_metrics_clean.parquet", index=False)


In [3]:
pd.read_parquet("../data/processed/company_metrics_clean.parquet")

,ticker,forwardPE,ev_to_ebitda,ebitda_margin,sector,industry,business_summary,ebitda,total_cash,total_debt,shares_outstanding,debt_to_ebitda
0,NVDA,17.739650,35.888,0.61698,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...,1.332300e+11,6.255600e+10,1.141200e+10,2.430000e+10,0.085656
1,GOOGL,25.076876,26.757,0.37279,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...,1.501750e+11,1.268430e+11,6.699600e+10,5.822000e+09,0.446119
2,AAPL,28.615034,25.736,0.35100,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ...",1.529020e+11,6.690700e+10,9.050900e+10,1.468114e+10,0.591941
3,MSFT,21.752918,17.616,0.57377,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...,1.752590e+11,8.946200e+10,1.232780e+11,7.425629e+09,0.703405
4,AMZN,26.442839,18.686,0.20327,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of...",1.457310e+11,1.230290e+11,1.785470e+11,1.075425e+10,1.225182
...,...,...,...,...,...,...,...,...,...,...,...,...
6040,RDAR,14.022167,0.000,-0.23284,Technology,Software - Application,"Telvantis Inc., provides technology products a...",-1.674758e+06,0.000000e+00,0.000000e+00,7.996261e+09,-0.000000
6041,QH,-9.400000,-1.879,-0.03605,Technology,Software - Application,"Quhuo Limited, through its subsidiaries, opera...",-9.222400e+07,3.088200e+07,1.238270e+08,2.519140e+07,-1.342677
6042,ECIA,-1.325444,0.000,-0.03686,Healthcare,Medical Instruments & Supplies,"Encision Inc., a medical device company, desig...",-2.322380e+05,0.000000e+00,0.000000e+00,1.687964e+07,-0.000000
6043,VCNX,-8.285714,-0.198,0.00000,Healthcare,Biotechnology,"Vaccinex, Inc., a clinical-stage biotechnology...",-1.864100e+07,1.107000e+06,2.500000e+04,2.676637e+06,-0.001341
